# Hybrid Replay (HR) for FCIL — reproduction runner (Colab GPU)

Reproduces *Federated Class-Incremental Learning: A Hybrid Approach Using Latent Exemplars and
Data-Free Techniques to Address Local and Global Forgetting* (Khademi Nori, Kim, Wang — ICLR 2025).

**How to use**
1. `Runtime → Change runtime type → GPU` (any GPU works; A100/L4 are ~3× faster than T4).
2. `Runtime → Run all`. Approve the Google Drive mount when asked.
3. Leave it running. Results are written to `MyDrive/hr_fcil_results/` after every task, and
   every job resumes from its checkpoint if the runtime disconnects: just run all cells again.

The job list lives in `configs/queue_colab.txt` in the GitHub branch and is re-pulled every
10 minutes, so new experiments can be queued from the repository without touching this notebook.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/hr_fcil_results'
import os; os.makedirs(RESULTS, exist_ok=True); print(RESULTS)

In [ ]:
REPO = 'https://github.com/miladkhademinori/autoencoder-hybrid-replay-cil.git'
BRANCH = 'claude/brave-ptolemy-2605ak'
import os
if not os.path.isdir('/content/hr'):
    !git clone -q -b {BRANCH} {REPO} /content/hr
else:
    !git -C /content/hr pull -q --ff-only
!git -C /content/hr log --oneline -3

In [ ]:
# CIFAR-100 (and TinyImageNet on demand) are downloaded into the Colab VM, not into Drive.
%cd /content/hr
!python -c "from hr_fcil.data import load_cifar100; x=load_cifar100('/content/data'); print([a.shape for a in x])"

In [ ]:
# Quick GPU throughput check (written to Drive so progress can be estimated remotely).
import socket; HOST = socket.gethostname()
!python scripts/bench.py --out {RESULTS}/bench_{HOST}.json

In [ ]:
# Run the queue. PARALLEL=2 runs two jobs side by side, which uses the GPU better
# (batch size 32 leaves a single job CPU/launch-bound). Set PARALLEL=1 if you hit OOM.
PARALLEL = 2
!python scripts/run_queue.py --queue configs/queue_colab.txt --results_root {RESULTS} --data_root /content/data --amp --parallel {PARALLEL} --git_pull_minutes 10 --idle_exit_minutes 60

In [ ]:
# Summary of everything finished so far
!python scripts/aggregate.py --results_root {RESULTS}